# Setup — Create Silver Lakehouse Tables
Creates all Silver 1 (staging) and Silver 2 (combined) Delta tables in the Lakehouse.

Safe to re-run — uses `mode('ignore')` so existing tables and data are never touched.

**Run once** after provisioning the Lakehouse, before triggering any pipeline DAGs.

In [ ]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, DateType, TimestampType,
)


### Parameters


In [ ]:
# Set these to match your Fabric workspace before running
WORKSPACE = "fabric_test_2"   # workspace_name from dev.yaml
LAKEHOUSE = "fabric_lakehouse"   # lakehouse from dev.yaml


### Setup


In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

spark = SparkSession.builder.getOrCreate()

print(f"[setup/create_silver_tables] workspace={WORKSPACE} lakehouse={LAKEHOUSE}")


### Helper

In [ ]:
def create_table(table_name: str, schema: StructType) -> None:
    """Write an empty DataFrame with the given schema as a Delta table.
    mode('ignore') means: create if not exists, skip if it already exists.
    """
    path = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{table_name}")
    (
        spark.createDataFrame([], schema)
        .write
        .format("delta")
        .mode("ignore")
        .save(path)
    )
    print(f"  OK  {table_name}")

### Silver 1 — Staging tables
Minimal schema with metadata columns only. Data columns are added automatically
by schema evolution when `clean.ipynb` writes the first real batch.

In [ ]:
print("[1/2] Silver 1 staging tables")

meta = [
    StructField("md_source_name", StringType(),   True),
    StructField("md_source_file", StringType(),   True),
    StructField("md_run_id",      StringType(),   True),
    StructField("md_updated_at",  TimestampType(), True),
]

create_table("staging_fedex_rate_card", StructType([
    StructField("effective_date",     DateType(),   True),
    StructField("expiry_date",        DateType(),   True),
    StructField("base_rate",          DoubleType(), True),
    StructField("fuel_surcharge_pct", DoubleType(), True),
] + meta))

### Silver 2 — Combined tables
Full canonical schemas matching the output of `combine.ipynb`.

In [ ]:
print("[2/2] Silver 2 combined tables")

create_table("silver_carrier_rate_card", StructType([
    StructField("carrier",             StringType(),    True),
    StructField("service_type",        StringType(),    True),
    StructField("zone",                StringType(),    True),
    StructField("weight_break",        DoubleType(),    True),
    StructField("base_rate",           DoubleType(),    True),
    StructField("fuel_surcharge_pct",  DoubleType(),    True),
    StructField("effective_date",      DateType(),      True),
    StructField("expiry_date",         DateType(),      True),
    StructField("md_source_name",      StringType(),    True),
    StructField("md_source_file",      StringType(),    True),
    StructField("md_run_id",           StringType(),    True),
    StructField("md_updated_at",       TimestampType(), True),
]))

In [ ]:
print("\nAll silver tables created (or already existed — no data was changed).")
mssparkutils.notebook.exit("done")  # noqa: F821